In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/Kronos_TW"
import os; os.makedirs(DRIVE_BASE, exist_ok=True)

In [ ]:
import os, subprocess, shutil
REPO = "/content/Kronos"
if not os.path.exists(REPO):
    subprocess.run([
        "git", "clone", "--branch", "feature/atr-vol-open-ic",
        "https://github.com/j835111/tw-quant-kronos", REPO
    ], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "origin", "feature/atr-vol-open-ic"], check=True)
os.chdir(REPO)

DRIVE_BASE = "/content/drive/MyDrive/Kronos_TW"
for sub in ["data", "outputs"]:
    local = f"finetune_tw/{sub}"
    remote = f"{DRIVE_BASE}/{sub}"
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        if os.readlink(local) == remote:
            print(f"{local} already correct")
            continue
        os.unlink(local)
    elif os.path.exists(local):
        shutil.copytree(local, remote, dirs_exist_ok=True)
        shutil.rmtree(local)
    os.symlink(remote, local)
    print(f"Symlinked {local} -> {remote}")
print("Symlinks ready.")


In [ ]:
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "yfinance", "pyyaml", "tqdm"], check=True)
print("Dependencies installed.")

In [ ]:
# Incremental update (--update skips symbols already downloaded)
import os
db = "finetune_tw/data/tw_stocks.db"
if not os.path.exists(db):
    subprocess.run([
        "python", "-m", "finetune_tw.download_data",
        "--config", "finetune_tw/configs/config_tw_daily_t4.yaml",
        "--source", "auto", "--start", "2015-01-01",
    ], check=True)
else:
    subprocess.run([
        "python", "-m", "finetune_tw.download_data",
        "--config", "finetune_tw/configs/config_tw_daily_t4.yaml", "--update",
    ], check=True)
print("Data ready.")


In [ ]:
# Download fine-tuned tokenizer from HF round-0 (skip train_tokenizer)
import os
tok_path = "finetune_tw/outputs/tw_daily/tokenizer/best_model"
if not os.path.exists(tok_path):
    print("Downloading tokenizer from HF j835111/kronos-tw-finetune@round-0 ...")
    from huggingface_hub import snapshot_download
    import shutil
    snap = snapshot_download(
        "j835111/kronos-tw-finetune",
        revision="round-0",
        allow_patterns=["tokenizer/best_model/*"],
        local_dir="/tmp/hf_round0_tok"
    )
    os.makedirs(os.path.dirname(tok_path), exist_ok=True)
    shutil.copytree(f"{snap}/tokenizer/best_model", tok_path)
    print(f"Tokenizer saved to {tok_path}")
else:
    print(f"Tokenizer already exists at {tok_path}")


In [ ]:
# Train predictor — auto-resumes from checkpoint if session restarted
subprocess.run([
    "python", "-m", "finetune_tw.train_predictor",
    "--config", "finetune_tw/configs/config_tw_daily_t4.yaml",
], check=True)


In [ ]:
subprocess.run([
    "python", "-m", "finetune_tw.backtest_next_open",
    "--config", "finetune_tw/configs/config_tw_daily_t4.yaml",
    "--model", "round-3",
    "--top_k", "10",
    "--hold_days_list", "5",
    "--atr-weights",
], check=True)
